Back Testing

In [2]:
from datetime import date, datetime
import pandas as pd
import yfinance as yf
import time
#from Alert import Alert
pd.options.mode.chained_assignment = None  # default='warn'
#alert = Alert('1h','GGAL')

input:
    
     Titulo  : example: GGAL
     frequencia de tick : 1h (frequencia mais alta)
     dftitulo : vista do BD sqtitulosalpha.bd (testar ultimos periodas)
Parametros a variar para back testing ( definir rangos de variação)
     k, d, smooth (parametros do stch)
     dayM  (frequencia media) (multiplicador da frequencia mais alta) exemplo: 5
     semM  (frequencia baixa) (multiplicador da frequencia media) exemplo: 7

In [4]:
dataini = '2022-04-03 19:30:00'
datafim = '2023-04-03 19:30:00'

In [5]:
import sqlite3
import pandas as pd

# Caminho para o banco de dados
caminho_bd = r'C:\Users\scitr\anaconda_projects\Trading_System\Dados_Fontes\Alpha_Vantage\sqtitulosalpha.db'

# Conectando ao banco
conexao = sqlite3.connect(caminho_bd)

# Lendo a view
#consulta = 'SELECT * FROM vwtitulosdados ORDER BY datetime'
consulta = f"""
SELECT * FROM vwtitulosdados
WHERE datetime BETWEEN '{dataini}' AND '{datafim}'
ORDER BY datetime
"""

dftitulosdados = pd.read_sql_query(consulta, conexao)

# Fechando a conexão
conexao.close()

# Exibindo os primeiros registros para conferir
display(dftitulosdados)

,symbol,moeda,intervalo,datetime,open,high,low,close,volume
0,GGAL,USD,60min,2022-04-04 09:00:00,8.9836,9.1457,8.9755,9.0079,45436.0
1,GGAL,USD,60min,2022-04-04 10:00:00,9.0484,9.0809,8.9755,8.9917,49030.0
2,GGAL,USD,60min,2022-04-04 11:00:00,8.9836,8.9998,8.9268,8.9268,41129.0
3,GGAL,USD,60min,2022-04-04 12:00:00,8.9268,9.0160,8.9268,8.9512,34248.0
4,GGAL,USD,60min,2022-04-04 13:00:00,8.9593,8.9593,8.8863,8.9187,67152.0
...,...,...,...,...,...,...,...,...,...
2101,GGAL,USD,60min,2023-04-03 12:00:00,9.5499,9.6357,9.5499,9.5756,29258.0
2102,GGAL,USD,60min,2023-04-03 13:00:00,9.5671,9.6872,9.5671,9.6700,48527.0
2103,GGAL,USD,60min,2023-04-03 14:00:00,9.6785,9.7447,9.6529,9.7301,32877.0
2104,GGAL,USD,60min,2023-04-03 15:00:00,9.7386,9.7816,9.6529,9.7558,143309.0


In [6]:

i = 'high'
K = 16  
D = 5    
smoth = 5  
dayM = 8  
semM = 4


In [7]:
# Stochastic calculation
def stochastic(df, i, K, D, smoth):
        
    df["k"] = (100. * (df.close - df.low.rolling(K).min()) /
        (df.high.rolling(K).max() - df.low.rolling(K).min()))
    
    df["k" + i] = df.k.rolling(smoth).mean()
    df["d" + i] = df["k" + i].rolling(D).mean()
    
    df.drop(columns=["k"], inplace=True)  

    return df
dfstoch = stochastic(dftitulosdados, i, K, D, smoth)
display (dfstoch)

,symbol,moeda,intervalo,datetime,open,high,low,close,volume,khigh,dhigh
0,GGAL,USD,60min,2022-04-04 09:00:00,8.9836,9.1457,8.9755,9.0079,45436.0,NaN,NaN
1,GGAL,USD,60min,2022-04-04 10:00:00,9.0484,9.0809,8.9755,8.9917,49030.0,NaN,NaN
2,GGAL,USD,60min,2022-04-04 11:00:00,8.9836,8.9998,8.9268,8.9268,41129.0,NaN,NaN
3,GGAL,USD,60min,2022-04-04 12:00:00,8.9268,9.0160,8.9268,8.9512,34248.0,NaN,NaN
4,GGAL,USD,60min,2022-04-04 13:00:00,8.9593,8.9593,8.8863,8.9187,67152.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2101,GGAL,USD,60min,2023-04-03 12:00:00,9.5499,9.6357,9.5499,9.5756,29258.0,35.345039,24.486408
2102,GGAL,USD,60min,2023-04-03 13:00:00,9.5671,9.6872,9.5671,9.6700,48527.0,43.814248,30.270650
2103,GGAL,USD,60min,2023-04-03 14:00:00,9.6785,9.7447,9.6529,9.7301,32877.0,53.958531,37.149336
2104,GGAL,USD,60min,2023-04-03 15:00:00,9.7386,9.7816,9.6529,9.7558,143309.0,64.423364,45.395125


In [8]:
def set_test( df, intervalo, k, d, smth, dayM, semM):
        
        # Convert time
        #df["time"] = pd.to_datetime(df.index, utc=True)
        #df['timeArg'] = df['time'].dt.tz_convert('America/Argentina/Buenos_Aires')        
        #df['time'] = df['time'].dt.tz_convert(None)

        df = stochastic(df, "high", k, d, smth)
        df = stochastic(df, "med", k*dayM, d*dayM, smth*dayM)
        df = stochastic(df, "low", k*dayM*semM, d*dayM*semM, smth*dayM*semM)
    
        return df
dfstoch_hml= set_test(dfstoch, i , K, D, smoth, dayM , semM )
%time set_test(dfstoch, i , K, D, smoth, dayM , semM ) 

CPU times: total: 15.6 ms
Wall time: 15 ms


,symbol,moeda,intervalo,datetime,open,high,low,close,volume,khigh,dhigh,kmed,dmed,klow,dlow
0,GGAL,USD,60min,2022-04-04 09:00:00,8.9836,9.1457,8.9755,9.0079,45436.0,NaN,NaN,NaN,NaN,NaN,NaN
1,GGAL,USD,60min,2022-04-04 10:00:00,9.0484,9.0809,8.9755,8.9917,49030.0,NaN,NaN,NaN,NaN,NaN,NaN
2,GGAL,USD,60min,2022-04-04 11:00:00,8.9836,8.9998,8.9268,8.9268,41129.0,NaN,NaN,NaN,NaN,NaN,NaN
3,GGAL,USD,60min,2022-04-04 12:00:00,8.9268,9.0160,8.9268,8.9512,34248.0,NaN,NaN,NaN,NaN,NaN,NaN
4,GGAL,USD,60min,2022-04-04 13:00:00,8.9593,8.9593,8.8863,8.9187,67152.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2101,GGAL,USD,60min,2023-04-03 12:00:00,9.5499,9.6357,9.5499,9.5756,29258.0,35.345039,24.486408,45.055766,33.328487,48.012866,70.547223
2102,GGAL,USD,60min,2023-04-03 13:00:00,9.5671,9.6872,9.5671,9.6700,48527.0,43.814248,30.270650,45.854905,33.918139,47.710336,70.311149
2103,GGAL,USD,60min,2023-04-03 14:00:00,9.6785,9.7447,9.6529,9.7301,32877.0,53.958531,37.149336,46.699935,34.512110,47.428408,70.073438
2104,GGAL,USD,60min,2023-04-03 15:00:00,9.7386,9.7816,9.6529,9.7558,143309.0,64.423364,45.395125,47.601915,35.110265,47.142843,69.834096


In [10]:

def long_buy_crits( df):
    
    df["longbuylow"] = 0
    df["longbuymed"] = 0
    df["longbuyhigh"] = 0
    df["longenter"] = 0 
    df["buylong"] = 0
    for i in df.index:    
        
        if df.loc[i,"klow"] > 20 and df.loc[i,"klow"] > df.loc[i,"dlow"] :
            df.loc[i, "longbuylow"] = 1
        else :
            df.loc[i, "longbuylow"] = 0
            
        if df.loc[i,"kmed"] > 20 and df.loc[i,"kmed"] > df.loc[i,"dmed"] :
            df.loc[i, "longbuymed"] = 1
        else :
            df.loc[i, "longbuymed"] = 0
    
        if df.loc[i,"khigh"] > 20 and df.loc[i,"khigh"] > df.loc[i,"dhigh"] :
            df.loc[i, "longbuyhigh"] = 1
        else :
            df.loc[i, "longbuyhigh"] = 0

        if  df.loc[i, "longbuyhigh"] == 1 and df.loc[i, "longbuymed"] == 1 and df.loc[i, "longbuylow"] == 1 :        
            df.loc[i, "longenter"] = 1
            df.loc[i, "buylong"] = 1
        else:
            df.loc[i, "longenter"] = 0
            df.loc[i, "buylong"] = 0
    return df

%time long_buy_crits(dfstoch_hml)   


CPU times: total: 3.56 s
Wall time: 3.62 s


,symbol,moeda,intervalo,datetime,open,high,low,close,volume,khigh,dhigh,kmed,dmed,klow,dlow,longbuylow,longbuymed,longbuyhigh,longenter,buylong
0,GGAL,USD,60min,2022-04-04 09:00:00,8.9836,9.1457,8.9755,9.0079,45436.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
1,GGAL,USD,60min,2022-04-04 10:00:00,9.0484,9.0809,8.9755,8.9917,49030.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2,GGAL,USD,60min,2022-04-04 11:00:00,8.9836,8.9998,8.9268,8.9268,41129.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
3,GGAL,USD,60min,2022-04-04 12:00:00,8.9268,9.0160,8.9268,8.9512,34248.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
4,GGAL,USD,60min,2022-04-04 13:00:00,8.9593,8.9593,8.8863,8.9187,67152.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2101,GGAL,USD,60min,2023-04-03 12:00:00,9.5499,9.6357,9.5499,9.5756,29258.0,35.345039,24.486408,45.055766,33.328487,48.012866,70.547223,0,1,1,0,0
2102,GGAL,USD,60min,2023-04-03 13:00:00,9.5671,9.6872,9.5671,9.6700,48527.0,43.814248,30.270650,45.854905,33.918139,47.710336,70.311149,0,1,1,0,0
2103,GGAL,USD,60min,2023-04-03 14:00:00,9.6785,9.7447,9.6529,9.7301,32877.0,53.958531,37.149336,46.699935,34.512110,47.428408,70.073438,0,1,1,0,0
2104,GGAL,USD,60min,2023-04-03 15:00:00,9.7386,9.7816,9.6529,9.7558,143309.0,64.423364,45.395125,47.601915,35.110265,47.142843,69.834096,0,1,1,0,0
